# 03 Evaluate: how good are the forecasts in the backtest?

**Purpose:** Compute the backtest errors of all candidate models, per horizon, in units a planner understands; compare the models at the main horizon ($h = 2$), also without the crisis periods and year by year; check whether the route indicator helps.
**Input:** `../data/processed/backtest_forecasts.csv` (from `02_estimate.ipynb`), `../data/processed/backtest_forecasts_regression.csv` (from `02b_estimate_regression.ipynb`) and `../data/processed/backtest_forecasts_indicator.csv` (from `02c_estimate_indicator.ipynb`).
**Output:** `../output/tables/03_backtest_errors.csv`, `../output/tables/03_backtest_errors_by_year_h2.csv`, `../output/tables/03_indicator_effect.csv` and two figures in `../output/figures/`.

This notebook uses **only the backtest** (2013 to September 2024). The test period stays untouched until we have chosen our final model (notebook 209, Step 7: choose before you test).

**Timeline.** We wrote the choice rule and chose our final model on 17 November 2026 (section 8, commit `c6a3976`), then ran the test once (`04_test.ipynb`, commit `4f284f5`). On 23 November we added one more simple rule, the **random walk** (`02_estimate.ipynb`, suggested by Axel). It appears in the tables below, but the main comparison of our plan (the best of our two planned baselines against the best model, section 3) and the choice in section 8 are unchanged. Section 9 compares our chosen model with the random walk.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
plt.style.use("seaborn-v0_8-muted")

## 1. Load the forecasts and compute the errors

For every forecast we compute two errors:

- the **absolute error in persons**, $|y - \hat y|$: the mean of it (MAE) says by how many applications per month we miss on average. This is the number a planner at SEM understands (notebook 108b, deck 08; notebook 209, Step 8).
- the **squared error in logs**, $(\log y - \widehat{\log y})^2$: the root of its mean (RMSE in logs) is the error measure of our project plan. An error of 0.10 in logs is roughly 10% off (notebook 209).

In [ ]:
# The forecasts of the three estimation notebooks, stacked one below the other (pd.concat, notebook 04)
forecasts_sarima = pd.read_csv("../data/processed/backtest_forecasts.csv")
forecasts_regression = pd.read_csv("../data/processed/backtest_forecasts_regression.csv").drop(columns=["alpha"])
forecasts_indicator = pd.read_csv("../data/processed/backtest_forecasts_indicator.csv").drop(columns=["alpha"])
forecasts = pd.concat([forecasts_sarima, forecasts_regression, forecasts_indicator], ignore_index=True)
forecasts["month"] = pd.to_datetime(forecasts["month"])
forecasts["origin"] = pd.to_datetime(forecasts["origin"])

forecasts["abs_error"] = (forecasts["actual"] - forecasts["forecast"]).abs()
forecasts["sq_error_log"] = (forecasts["actual_log"] - forecasts["forecast_log"]) ** 2

print("Forecasts:", forecasts.shape[0], "| models:", forecasts["model"].nunique(),
      "| forecast months from", forecasts["month"].min(), "to", forecasts["month"].max())

## 2. Errors of all models for all horizons

We put the three simple rules first, then the three (S)ARIMA models (`02_estimate.ipynb`), the three regression models on the series' own past (`02b_estimate_regression.ipynb`), and the same three regressions with the route indicator (`02c_estimate_indicator.ipynb`). All twelve models were evaluated on exactly the same forecast months (notebook 209: the same folds and the same error measure for all).

In [ ]:
model_order = ["Seasonal naive", "12-month average", "Random walk",
               "ARIMA(1,1,0)", "SARIMA(1,1,0)(1,0,0)12", "SARIMA(0,1,1)(0,1,1)12",
               "Linear regression", "Ridge", "Lasso",
               "Linear regression + routes", "Ridge + routes", "Lasso + routes"]
baselines = ["Seasonal naive", "12-month average"]      # the two baselines of our project plan
added_after = ["Random walk"]                               # added on 23 November, after the test run
sarima_models = ["ARIMA(1,1,0)", "SARIMA(1,1,0)(1,0,0)12", "SARIMA(0,1,1)(0,1,1)12"]
regression_models = ["Linear regression", "Ridge", "Lasso"]
indicator_models = ["Linear regression + routes", "Ridge + routes", "Lasso + routes"]

# One fixed colour per model, the same in every figure (plt.cm.tab10, notebook 211).
# A model "+ routes" gets the colour of the same model without the indicator (and a dashed line below).
colors = dict(zip(model_order[:9], plt.cm.tab10(np.arange(9))))
for name in regression_models:
    colors[name + " + routes"] = colors[name]

def error_table(data):
    # MAE in persons and RMSE in logs for every model and horizon
    grouped = data.groupby(["model", "h"])
    table = pd.DataFrame({
        "MAE (persons)": grouped["abs_error"].mean(),
        "RMSE (log)": np.sqrt(grouped["sq_error_log"].mean()),
        "forecasts": grouped["abs_error"].size(),
    }).reset_index()
    return table

errors = error_table(forecasts)

mae = errors.pivot(index="model", columns="h", values="MAE (persons)").loc[model_order].round(0)
mae.columns = [f"MAE h={h}" for h in mae.columns]
rmse = errors.pivot(index="model", columns="h", values="RMSE (log)").loc[model_order].round(3)
rmse.columns = [f"RMSE log h={h}" for h in rmse.columns]

overview = pd.concat([mae, rmse], axis=1)
overview.to_csv("../output/tables/03_backtest_errors.csv")
overview

**What we see.**
- At every horizon, the two simple rules of our plan have the largest errors: the seasonal naive about 680 persons, the 12-month average 487 to 558. All six models that use the series' own past are far below them.
- The **random walk** (added after the test, section 9) is much better than these two rules: 259, 379 and 487 persons, close to the models.
- These six models are close to each other. At $h = 2$, their MAE lies between 358 and 388 persons: the three regressions are at 358, 358 and 360, the best SARIMA at 369, ARIMA(1,1,0) at 388.
- The two error measures do not crown the same model. By MAE, Ridge is lowest at all three horizons (248, 358, 447). By RMSE in logs, SARIMA(0,1,1)(0,1,1)12 is lowest at $h = 2$ (0.274) and $h = 3$ (0.328), and Ridge at $h = 1$ (0.187).
- As before, the seasonal naive has almost the same error at every horizon, because its forecast is always the value 12 months earlier, whatever the origin.
- The three regressions **with the route indicator** (last three rows) are a little worse than the same regressions without it: at $h = 2$ their MAE is 364 to 374 against 358 to 360. We look at this in detail in section 7.

## 3. The main comparison: $h = 2$

Our project plan fixes one main comparison (Johannes' feedback): the **best simple baseline of our plan** (seasonal naive or 12-month average) against the **best model**, at the main horizon $h = 2$, by MAE in persons. We read all of them from the table above with code, so that nothing is chosen by hand. We also print the best model of each family.

In [ ]:
h2 = errors[errors["h"] == 2].set_index("model")

best_baseline = h2.loc[baselines, "MAE (persons)"].idxmin()
best_sarima = h2.loc[sarima_models, "MAE (persons)"].idxmin()
best_regression = h2.loc[regression_models, "MAE (persons)"].idxmin()
best_indicator = h2.loc[indicator_models, "MAE (persons)"].idxmin()
best_model = h2.loc[sarima_models + regression_models + indicator_models, "MAE (persons)"].idxmin()

for label, name in [("Best simple baseline", best_baseline), ("Best (S)ARIMA model", best_sarima),
                    ("Best regression model", best_regression), ("Best model with routes", best_indicator),
                    ("Best model overall", best_model)]:
    print(f"{label + ' at h=2:':28s} {name:27s} MAE {h2.loc[name, 'MAE (persons)']:5.1f} persons   "
          f"RMSE (log) {h2.loc[name, 'RMSE (log)']:.3f}")

mae_base, mae_best = h2.loc[best_baseline, "MAE (persons)"], h2.loc[best_model, "MAE (persons)"]
rmse_base, rmse_best = h2.loc[best_baseline, "RMSE (log)"], h2.loc[best_model, "RMSE (log)"]
print()
print(f"Improvement of the best model over the best baseline: {100 * (1 - mae_best / mae_base):.0f}% lower MAE, "
      f"{100 * (1 - rmse_best / rmse_base):.0f}% lower RMSE (log)")
average_actual = forecasts[forecasts["h"] == 2]["actual"].mean()
print(f"For comparison, average actual number of applications in these months: {average_actual:.0f}")

**What we see.**
- The best simple baseline of our plan is the 12-month average (MAE 527.7 persons, RMSE 0.342). The best model overall, by MAE, is Ridge (357.9 persons, RMSE 0.279): **32% lower MAE** and 18% lower RMSE (log) than the 12-month average. For comparison, there were on average 1,850 applications per month in these months.
- Ridge is only 11 applications per month better than the best SARIMA model (369.0): about 3%. By RMSE (log), the SARIMA model is slightly better (0.274 against 0.279). The plain linear regression has practically the same MAE as Ridge (358 in the table above).
- The best model with the route indicator is Ridge + routes (364.1), so adding the indicator does not change the best model.

**What this means for us.** Using the series' own past clearly beats the simple rules of our plan, by far more than the 10% of our success rule. (Against the random walk, added later, the gain is much smaller: section 9.) Between the regressions and SARIMA, there is no clear winner: Ridge's gain over SARIMA is far below 10%, and the other error measure points the other way. The six own-history models, at about 358 to 369 persons, are the benchmark that an indicator block must beat.

## 4. Without the crisis periods

The backtest contains two exceptional periods: the 2015-16 crisis and the drop of 2020. As planned, we report the errors also without the forecast months of these years (2015-01 to 2016-12 and 2020-01 to 2020-12), and for the recent years only (from 2022). These are **evaluation periods**: the models themselves did not know about them.

In [ ]:
month = forecasts["month"]
crisis = ((month >= "2015-01-01") & (month <= "2016-12-01")) | ((month >= "2020-01-01") & (month <= "2020-12-01"))

periods = {
    "all backtest months": forecasts,
    "without 2015-16 and 2020": forecasts[~crisis],
    "from 2022": forecasts[month >= "2022-01-01"],
}

rows = []
for period_name, data in periods.items():
    table = error_table(data[data["h"] == 2])
    table["period"] = period_name
    rows.append(table)
by_period = pd.concat(rows)

by_period.pivot(index="model", columns="period", values="MAE (persons)").loc[model_order, list(periods)].round(0)

In [ ]:
# Number of forecast months in each period (h = 2)
by_period[by_period["model"] == "Seasonal naive"].set_index("period")["forecasts"]

**What we see.** (The three periods contain 140, 104 and 33 forecast months.)
- Without 2015-16 and 2020, all errors are smaller. The best own-history models are SARIMA(1,1,0)(1,0,0)12 (281) and Ridge (282), against 369 for the 12-month average: the gap to the simple rule shrinks from 170 to 88 persons. The random walk is at 309.
- From 2022, the linear regression (413) and Ridge (415) are lowest, before the best SARIMA (427), the 12-month average is at 512 and the random walk at 488.
- The models with the route indicator are a little worse than their versions without it in the first two periods. From 2022, the linear regression + routes is equal to the linear regression (412 against 413), while Ridge and Lasso + routes are worse (436 and 431).

**What this means for us.** The best own-history model changes from one period to the next: the regressions and the (S)ARIMA models trade places. Their differences are small compared with how much the errors change between periods.

## 5. Is the difference real, or within the noise? Year by year

Notebook 209 compares models fold by fold: each validation fold gives one error, and we look at whether the gap between two models is larger than the spread across the folds, and whether one model wins in most folds. In a time-series backtest, the natural "folds" are the **years**: we compute the MAE at $h = 2$ for each year of forecast months (2013 to 2024; 2013 starts in February and 2024 ends in September).

*Not seen in class:* `forecasts["month"].dt.year` gives the year of each forecast month.

In [ ]:
h2_all = forecasts[forecasts["h"] == 2].copy()
h2_all["year"] = h2_all["month"].dt.year

by_year = h2_all.groupby(["year", "model"])["abs_error"].mean().reset_index()
by_year = by_year.pivot(index="year", columns="model", values="abs_error")[model_order].round(0)
by_year.columns.name = None
by_year.to_csv("../output/tables/03_backtest_errors_by_year_h2.csv")
by_year

In [ ]:
# Two comparisons, year by year: (1) best model vs best baseline, (2) best regression vs best (S)ARIMA
for a, b in [(best_model, best_baseline), (best_regression, best_sarima)]:
    wins = (by_year[a] < by_year[b]).sum()
    print(f"{a} has a lower MAE than {b} in {wins} of {by_year.shape[0]} years.")
    print(f"   Spread across years (standard deviation of the yearly MAE): {b} {by_year[b].std():.0f}, {a} {by_year[a].std():.0f}")
    print(f"   Gap between the two averages: {by_year[b].mean() - by_year[a].mean():.0f}")

The picture follows notebook 209 (Step 5): each dot is one year, the black bar is the average over the years.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6.5))
for i, name in enumerate(model_order):
    errors_by_year = by_year[name]
    ax.scatter(errors_by_year, [i] * len(errors_by_year), alpha=0.6, color=colors[name])
    ax.scatter([errors_by_year.mean()], [i], marker="|", s=600, color="black")
ax.set_yticks(range(len(model_order)))
ax.set_yticklabels(model_order)
ax.invert_yaxis()
ax.set_xlim(left=0)
ax.set_xlabel("Mean absolute error per year, h = 2 (applications per month)")
ax.set_title("Backtest errors two months ahead: one dot per year (2013-2024)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("../output/figures/03_errors_by_year_h2.png")
plt.show()

**What we see.**
- **Best model against best baseline:** Ridge has a lower MAE than the 12-month average in 8 of 12 years. The gap between their averages (167) is smaller than the spread across years (standard deviation 417 and 220). In the picture, the two dots far to the right of the 12-month average (about 1,190 and 1,500) are the years 2016 and 2015 (table above).
- **Best regression against best SARIMA:** Ridge is lower in 9 of 12 years, but the gap between the averages is only 11 persons, against a spread across years of about 220. Several of its wins are tiny (2019: 135 against 136; 2013: 207 against 216), and it loses in 2016, 2020 and 2021.
- The random walk (third row) has its largest errors in 2015, 2016 and 2023 (670, 785, 729), like the models, and is close to them in the other years.
- In the picture, the six own-history models look almost the same: their averages (black bars) lie close together, between about 358 and 388. The three rows "+ routes" look very similar to the rows of the same models without the indicator.

**What this means for us.** The advantage of the own-history models over the simple rules is consistent, but it comes mostly from the years with big changes. Among the six own-history models, the differences are within the noise. When we choose the final model before the test period, a simpler model within the noise is a serious candidate (notebook 209, Step 7).

## 6. Errors over the horizon

How fast do the errors grow when we forecast further ahead? We plot the MAE of each model for $h = 1, 2, 3$.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for i, name in enumerate(model_order):
    one = errors[errors["model"] == name]
    style = "--" if name in indicator_models else "-"
    ax.plot(one["h"], one["MAE (persons)"], marker="o", linestyle=style, color=colors[name], label=name)
ax.set_xticks([1, 2, 3])
ax.set_xlabel("Horizon h (months ahead)")
ax.set_ylabel("MAE (applications per month)")
ax.set_title("Backtest MAE by horizon")
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../output/figures/03_mae_by_horizon.png")
plt.show()

**What we see.** For the six own-history models, the MAE grows steadily from about 250 at $h = 1$ to about 450 to 500 at $h = 3$, as expected: the further ahead, the less the recent months tell us (notebook 202). Ridge is lowest at every horizon (248, 358, 447), but the lines of the regressions and the SARIMA models almost overlap. The dashed lines (models with the route indicator) lie slightly above the lines of the same models without it, most visibly at $h = 3$. ARIMA(1,1,0), without a seasonal part, falls behind at $h = 3$ (499), close to the random walk (259, 379, 487). The 12-month average grows more slowly (487 to 558), and the seasonal naive stays flat at about 680.

## 7. Does the route indicator help?

Our project plan adds the indicator block **one block at a time**: the models of `02c` are exactly those of `02b` plus two route features. So we compare each regression with its own "+ routes" version, on the same forecast months. We use the rule of our project plan: a change counts as a clear improvement only if it lowers the MAE by at least 10%.

In [ ]:
rows = []
for name in regression_models:
    for h in [1, 2, 3]:
        without = errors[(errors["model"] == name) & (errors["h"] == h)].iloc[0]
        with_routes = errors[(errors["model"] == name + " + routes") & (errors["h"] == h)].iloc[0]
        rows.append({
            "model": name, "h": h,
            "MAE without routes": without["MAE (persons)"],
            "MAE with routes": with_routes["MAE (persons)"],
            "change in MAE (%)": 100 * (with_routes["MAE (persons)"] / without["MAE (persons)"] - 1),
            "RMSE (log) without": without["RMSE (log)"],
            "RMSE (log) with": with_routes["RMSE (log)"],
        })
indicator_effect = pd.DataFrame(rows).set_index(["model", "h"])
indicator_effect.to_csv("../output/tables/03_indicator_effect.csv")
indicator_effect.round(3)

**What we see.**
- At the main horizon $h = 2$, all three regressions get **worse** with the route indicator: the MAE goes up by 1.7% (Ridge), 3.8% (Lasso) and 4.2% (linear regression). At $h = 3$ the loss is larger (3.8% to 7.0%).
- The only gain is Lasso at $h = 1$: 250.4 instead of 254.4 persons (−1.6%), and also a slightly lower RMSE (log) (0.190 against 0.193). Ridge and the linear regression are worse at $h = 1$ too.
- The RMSE (log) tells the same story: with the routes it is higher in 8 of the 9 cases.
- No change comes close to the 10% improvement of our success rule.

**What this means for us.** The route indicator, as we can use it at the cutoff, does not improve the forecasts. This fits what we found in `01b_preprocess_indicators.ipynb`: the change on the routes moved together with the change in applications in the same month and one month later (correlations 0.31 and 0.35), but hardly at all from two months on (0.08 or less). Because Frontex is published one month later than the target, the newest route month we may use lies at least $h + 1$ months before the forecast month, so the useful part of the signal arrives too late.

The same comparison year by year and by period, at the main horizon $h = 2$ (as in sections 4 and 5).

In [ ]:
for name in regression_models:
    with_routes = name + " + routes"
    wins = (by_year[with_routes] < by_year[name]).sum()
    gap = by_year[name].mean() - by_year[with_routes].mean()
    print(f"{with_routes:27s} lower MAE than {name:18s} in {wins:2d} of {by_year.shape[0]} years | "
          f"gap of the averages {gap:6.1f} | spread across years {by_year[name].std():.0f}")
print()
by_period.pivot(index="model", columns="period", values="MAE (persons)").loc[regression_models + indicator_models, list(periods)].round(0)

In [ ]:
# Year by year at h = 2: MAE of Ridge without and with the routes, and the difference
ridge_years = by_year[["Ridge", "Ridge + routes"]].copy()
ridge_years["with minus without"] = ridge_years["Ridge + routes"] - ridge_years["Ridge"]
ridge_years

**What we see.**
- Year by year at $h = 2$, the versions with the routes are better in only 2 or 3 of the 12 years. The gaps of the averages (6 to 15 persons, in favour of the models *without* the routes) are small compared with the spread across years (about 210 to 220).
- For Ridge, the one year with a clear gain is **2015**, the start of the big crisis: 629 instead of 704 (−75). But in 2016 the routes made the forecasts worse (+48), and also in 2022 and 2023 (+23 and +32). In the other years the differences are small (between −2 and +19).
- Without 2015-16 and 2020, all three models are worse with the routes (291 to 297 against 282 to 288).

**What this means for us.** The loss from adding the routes is small and within the noise, but there is no sign of a gain either, except in 2015. One good year is not enough to choose a model (notebook 209: we look at all folds, not at the best one). Following our plan (one block at a time, keep it only if it helps), **we do not keep the route indicator in the final model**. We report this as a result: for forecasts two months ahead, the past of the Swiss series itself already contains what the routes tell us in time.

## 8. Our choice, before we look at the test period

Notebook 209 (Step 7): *choose before you test*, and write down the choice and the reason. The three best models at $h = 2$:

In [ ]:
candidates = ["Ridge", "Linear regression", "SARIMA(0,1,1)(0,1,1)12"]
overview.loc[candidates]

**Our choice: the linear regression** (the one of `02b`, on the series' own past, one model per horizon).

**Reason.**
- Ridge has the lowest MAE, but the linear regression is practically equal: 358.0 against 357.9 persons at $h = 2$ (0.06 persons, section 7 table), with a slightly lower RMSE (log) (0.278 against 0.279). At $h = 1$ and $h = 3$, Ridge is a little better (248 against 256, 447 against 452 persons); our plan fixes $h = 2$ as the main horizon. SARIMA(0,1,1)(0,1,1)12 is about 3% worse by MAE at $h = 2$. All these differences are far smaller than the spread across years (about 220, section 5).
- Notebook 209, Step 7: if a simpler model is within the spread, it is a serious candidate. The linear regression is simpler than Ridge: it has no penalty, so it needs no choice of `alpha` inside each window, and its coefficients are easy to explain to a planner.
- The route indicator is not used (section 7).

**What we test, once.** The linear regression and, for the main comparison fixed in our project plan, the best simple baseline (the 12-month average). The test period is 2024-10 to 2026-09; the forecasts are made in the same way as in the backtest: at each origin, the model is fitted on all months known at that origin. We do not change this choice after seeing the test results.

## 9. Added after the test: the random walk

The random walk ("next months = last month") is the simplest rule a planner could use. We added it on 23 November, after our choice and the test run, so it cannot change the choice. But it is an important check: is our model better than simply repeating the last known month? We compare the chosen linear regression with it in the same way as in sections 3 and 5: by horizon, and year by year at $h = 2$.

In [ ]:
rw = errors[errors["model"] == "Random walk"].set_index("h")
lr = errors[errors["model"] == "Linear regression"].set_index("h")
random_walk_check = pd.DataFrame({
    "MAE random walk": rw["MAE (persons)"],
    "MAE linear regression": lr["MAE (persons)"],
    "change in MAE (%)": 100 * (lr["MAE (persons)"] / rw["MAE (persons)"] - 1),
    "RMSE (log) random walk": rw["RMSE (log)"],
    "RMSE (log) linear regression": lr["RMSE (log)"],
})
random_walk_check.to_csv("../output/tables/03_random_walk.csv")
print(random_walk_check.round(3))
print()
wins = (by_year["Linear regression"] < by_year["Random walk"]).sum()
gap = by_year["Random walk"].mean() - by_year["Linear regression"].mean()
print(f"Linear regression lower than the random walk in {wins} of {by_year.shape[0]} years | "
      f"gap of the averages {gap:.0f} | spread across years (random walk) {by_year['Random walk'].std():.0f}")
print("Success rule (at least 10% lower MAE at h = 2):", random_walk_check.loc[2, "change in MAE (%)"] <= -10)

**What we see.**
- The chosen linear regression is better than the random walk at every horizon, but only a little: **5.4% lower MAE at $h = 2$** (358 against 379), 1.4% at $h = 1$ (256 against 259) and 7.2% at $h = 3$ (452 against 487). At $h = 1$, the random walk even has a slightly lower RMSE (log) (0.195 against 0.197).
- Year by year at $h = 2$, the linear regression is lower in 7 of 12 years; the gap of the averages (19 persons) is far smaller than the spread across years (237).
- Against the random walk, our success rule (at least 10% lower MAE) is **not met** in the backtest.

**What this means for us.** Most of what can be predicted two months ahead is already in the last known month. Our model adds the seasonal pattern and the partial reversal of recent jumps, which helps a little, more at longer horizons. We report this honestly: our model clearly beats the two baselines of our plan, but in the backtest it is not clearly better than repeating the last month. `04_test.ipynb` also reports the random walk on the test period, for information.

## Summary

- **Main comparison ($h = 2$):** best simple baseline = 12-month average (MAE 528 persons); best model = Ridge (MAE 358 persons): **32% lower MAE**, 18% lower RMSE (log).
- **Regressions against SARIMA:** practically equal. Ridge is 3% lower by MAE but slightly worse by RMSE (log); it wins in 9 of 12 years, mostly by small margins; the gap (11 persons) is tiny compared with the spread across years.
- **Robustness:** the own-history models beat the simple rules in all periods, but the advantage is smaller without 2015-16 and 2020, and the best own-history model changes with the period.
- **Route indicator (one block at a time):** adding the detections on the three routes to the regressions makes the MAE at $h = 2$ 1.7% to 4.2% higher; the models with the routes win in only 2 or 3 of 12 years. The only clear gain is in 2015. We do not keep the indicator: it is published too late to help at our cutoff.
- **Added after the test (section 9):** the random walk, at 379 persons, is only 5.4% worse than our model at $h = 2$ (7 of 12 years); against it, the success rule is not met in the backtest.
- **Final choice (before the test):** the linear regression on the series' own past. It is as good as Ridge (358.0 against 357.9 persons at $h = 2$) and simpler. It is evaluated once on the test period in `04_test.ipynb`, together with the 12-month average.